In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
class ScoreModel:
    def __init__(
        self,
        min_time_for_score=5 * 60,
        time_weight_unit=3600,
        max_time_weight=10.0,
        decay_half_life=30 * 60,
        max_score=10.0,
        priority_bandwidth_pct=90,
    ):
        self.min_time_for_score = min_time_for_score
        self.time_weight_unit = time_weight_unit
        self.max_time_weight = max_time_weight
        self.decay_half_life = decay_half_life
        self.max_score = max_score
        self.priority_bandwidth_pct = priority_bandwidth_pct

    def time_weight(self, time_known_sec):
        return min(time_known_sec / self.time_weight_unit, self.max_time_weight)

    def decay(self, idle_time_sec):
        return 0.5 ** (idle_time_sec / self.decay_half_life)

    def score(self, contributions, time_known_sec, idle_time_sec=0):
        if time_known_sec < self.min_time_for_score:
            return 0.0
        raw = contributions * self.time_weight(time_known_sec) * self.decay(idle_time_sec)
        return min(raw, self.max_score)

    def contributions_to_maintain_cap(self, time_known_sec, idle_time_sec):
        tw = self.time_weight(time_known_sec)
        d = self.decay(idle_time_sec)
        if tw * d == 0:
            return float('inf')
        return self.max_score / (tw * d)

    def bandwidth_share(self, my_score, total_score):
        if total_score == 0:
            return 0
        return (my_score / total_score) * self.priority_bandwidth_pct

    def identities_for_bandwidth(self, target_bw_pct, honest_total_score, attacker_score_per_id):
        if attacker_score_per_id == 0:
            return float('inf')
        target_ratio = target_bw_pct / self.priority_bandwidth_pct
        if target_ratio >= 1:
            return float('inf')
        attacker_score_needed = (target_ratio * honest_total_score) / (1 - target_ratio)
        return int(np.ceil(attacker_score_needed / attacker_score_per_id))

In [ ]:
model = ScoreModel(max_score=10.0)

hours = np.linspace(0, 2, 500)
time_sec = hours * 3600
contributions_list = [10, 50, 100, 500, 1000]

fig = go.Figure()
for c in contributions_list:
    scores = [model.score(c, t) for t in time_sec]
    fig.add_trace(go.Scatter(x=hours, y=scores, name=f'{c} contrib'))

fig.update_layout(
    title='Score Growth (No Idle Time)',
    xaxis_title='Time Known (hours)',
    yaxis_title='Score',
    height=400
)
fig.show()

In [ ]:
model = ScoreModel(max_score=10.0)
time_known = 10 * 3600

idle_hours = np.linspace(0, 4, 500)
idle_sec = idle_hours * 3600
contributions_list = [1, 2, 5, 10, 100]

fig = go.Figure()
for c in contributions_list:
    scores = [model.score(c, time_known, idle) for idle in idle_sec]
    fig.add_trace(go.Scatter(x=idle_hours, y=scores, name=f'{c} contrib'))

fig.add_hline(y=model.max_score, line_dash='dash', line_color='gray', annotation_text='Cap')

fig.update_layout(
    title=f'Score Decay (10h established, MAX_SCORE={model.max_score})',
    xaxis_title='Idle Time (hours)',
    yaxis_title='Score',
    height=400
)
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['MAX_SCORE=10', 'MAX_SCORE=100'])

for col, max_score in enumerate([10, 100], 1):
    model = ScoreModel(max_score=max_score)
    time_known = 10 * 3600
    idle_hours = np.linspace(0, 4, 100)
    
    for c in [1, 5, 10, 50, 100]:
        scores = [model.score(c, time_known, idle * 3600) for idle in idle_hours]
        fig.add_trace(
            go.Scatter(x=idle_hours, y=scores, name=f'{c} contrib', showlegend=(col==1)),
            row=1, col=col
        )
    fig.add_hline(y=max_score, line_dash='dash', line_color='gray', row=1, col=col)

fig.update_xaxes(title_text='Idle Time (hours)')
fig.update_yaxes(title_text='Score')
fig.update_layout(title='Score Decay Comparison', height=400)
fig.show()

In [ ]:
IP_COST_PER_MONTH = 5
TX_COST = 0.001

def attack_cost(
    model,
    honest_relays,
    honest_contributions,
    honest_age_sec,
    honest_idle_sec,
    attacker_contributions,
    attacker_age_sec,
    target_bw_pct,
):
    honest_score_per_id = model.score(honest_contributions, honest_age_sec, honest_idle_sec)
    honest_total_score = honest_relays * honest_score_per_id
    
    attacker_score_per_id = model.score(attacker_contributions, attacker_age_sec, 0)
    
    if attacker_score_per_id == 0 or honest_total_score == 0:
        return None
    
    ids_needed = model.identities_for_bandwidth(target_bw_pct, honest_total_score, attacker_score_per_id)
    
    if ids_needed == float('inf') or ids_needed > 1_000_000:
        return None
    
    ip_cost = ids_needed * IP_COST_PER_MONTH
    tx_cost = ids_needed * attacker_contributions * TX_COST
    
    return {
        'ids': ids_needed,
        'ip_cost': ip_cost,
        'tx_cost': tx_cost,
        'total_cost': ip_cost + tx_cost,
        'honest_score_per_id': honest_score_per_id,
        'honest_total_score': honest_total_score,
        'attacker_score_per_id': attacker_score_per_id,
    }

In [ ]:
model = ScoreModel(max_score=10.0)

HONEST_RELAYS = 1000
HONEST_CONTRIBUTIONS = 100
HONEST_AGE = 10 * 3600
ATTACKER_CONTRIBUTIONS = 100
ATTACKER_AGE = 1 * 3600
TARGET_BW = 80

honest_idle_hours = np.linspace(0, 6, 100)
costs = []
for idle_h in honest_idle_hours:
    result = attack_cost(
        model, HONEST_RELAYS, HONEST_CONTRIBUTIONS, HONEST_AGE, idle_h * 3600,
        ATTACKER_CONTRIBUTIONS, ATTACKER_AGE, TARGET_BW
    )
    costs.append(result['total_cost'] if result else None)

fig = go.Figure()
fig.add_trace(go.Scatter(x=honest_idle_hours, y=costs, name='Attack cost'))

fig.update_layout(
    title=f'Attack Cost vs Honest Idle Time ({HONEST_RELAYS} relays, {TARGET_BW}% target)',
    xaxis_title='Honest Relays Idle Time (hours)',
    yaxis_title='Total Cost ($)',
    height=400
)
fig.show()

In [ ]:
model = ScoreModel(max_score=10.0)

HONEST_RELAYS = 1000
HONEST_CONTRIBUTIONS = 100
HONEST_AGE = 10 * 3600
ATTACKER_CONTRIBUTIONS = 100
TARGET_BW = 80

honest_idle_scenarios = [
    ('Active (0 idle)', 0),
    ('30min idle', 30 * 60),
    ('1h idle', 1 * 3600),
    ('2h idle', 2 * 3600),
]

attacker_age_scenarios = [
    ('Instant (6min)', 6 * 60),
    ('1h prep', 1 * 3600),
    ('10h prep', 10 * 3600),
]

data = []
for honest_name, honest_idle in honest_idle_scenarios:
    for attacker_name, attacker_age in attacker_age_scenarios:
        result = attack_cost(
            model, HONEST_RELAYS, HONEST_CONTRIBUTIONS, HONEST_AGE, honest_idle,
            ATTACKER_CONTRIBUTIONS, attacker_age, TARGET_BW
        )
        if result:
            data.append((
                honest_name, attacker_name,
                result['ids'], result['total_cost'],
                result['honest_score_per_id'], result['attacker_score_per_id']
            ))

fig = go.Figure(data=[go.Table(
    header=dict(values=['Honest State', 'Attacker Prep', 'IPs Needed', 'Cost $', 'Honest Score', 'Attacker Score']),
    cells=dict(values=[
        [d[0] for d in data],
        [d[1] for d in data],
        [f'{d[2]:,}' for d in data],
        [f'${d[3]:,.0f}' for d in data],
        [f'{d[4]:.2f}' for d in data],
        [f'{d[5]:.2f}' for d in data],
    ])
)])

fig.update_layout(
    title=f'Attack Cost Matrix (MAX_SCORE={model.max_score}, {HONEST_RELAYS} relays, {TARGET_BW}% target)',
    height=500
)
fig.show()

In [ ]:
HONEST_RELAYS = 1000
HONEST_CONTRIBUTIONS = 100
HONEST_AGE = 10 * 3600
ATTACKER_CONTRIBUTIONS = 100
TARGET_BW = 80

fig = make_subplots(rows=1, cols=2, subplot_titles=['MAX_SCORE=10', 'MAX_SCORE=100'])

for col, max_score in enumerate([10, 100], 1):
    model = ScoreModel(max_score=max_score)
    
    for attacker_name, attacker_age in [('Instant (6min)', 6*60), ('1h prep', 3600), ('10h prep', 10*3600)]:
        honest_idle_hours = np.linspace(0, 6, 100)
        costs = []
        for idle_h in honest_idle_hours:
            result = attack_cost(
                model, HONEST_RELAYS, HONEST_CONTRIBUTIONS, HONEST_AGE, idle_h * 3600,
                ATTACKER_CONTRIBUTIONS, attacker_age, TARGET_BW
            )
            costs.append(result['total_cost'] if result else None)
        
        fig.add_trace(
            go.Scatter(x=honest_idle_hours, y=costs, name=attacker_name, showlegend=(col==1)),
            row=1, col=col
        )

fig.update_xaxes(title_text='Honest Idle Time (hours)')
fig.update_yaxes(title_text='Attack Cost ($)')
fig.update_layout(title=f'Attack Cost: MAX_SCORE=10 vs 100 ({TARGET_BW}% target)', height=400)
fig.show()

In [ ]:
HONEST_RELAYS = 1000
HONEST_AGE = 10 * 3600
ATTACKER_CONTRIBUTIONS = 100
ATTACKER_AGE = 1 * 3600
TARGET_BW = 80

fig = make_subplots(rows=1, cols=2, subplot_titles=['MAX_SCORE=10', 'MAX_SCORE=100'])

for col, max_score in enumerate([10, 100], 1):
    model = ScoreModel(max_score=max_score)
    
    for honest_contrib in [10, 50, 100, 500]:
        honest_idle_hours = np.linspace(0, 6, 100)
        costs = []
        for idle_h in honest_idle_hours:
            result = attack_cost(
                model, HONEST_RELAYS, honest_contrib, HONEST_AGE, idle_h * 3600,
                ATTACKER_CONTRIBUTIONS, ATTACKER_AGE, TARGET_BW
            )
            costs.append(result['total_cost'] if result else None)
        
        fig.add_trace(
            go.Scatter(x=honest_idle_hours, y=costs, name=f'{honest_contrib} contrib', showlegend=(col==1)),
            row=1, col=col
        )

fig.update_xaxes(title_text='Honest Idle Time (hours)')
fig.update_yaxes(title_text='Attack Cost ($)')
fig.update_layout(title=f'Attack Cost by Honest Contribution History ({TARGET_BW}% target)', height=400)
fig.show()

In [ ]:
model = ScoreModel(max_score=10.0)
time_known = 10 * 3600

idle_hours = [0, 0.5, 1, 2, 4]
contributions = [1, 2, 5, 10, 20, 50, 100]

data = []
for c in contributions:
    row = [c]
    for idle_h in idle_hours:
        s = model.score(c, time_known, idle_h * 3600)
        row.append(f'{s:.2f}')
    data.append(row)

fig = go.Figure(data=[go.Table(
    header=dict(values=['Contributions'] + [f'{h}h idle' for h in idle_hours]),
    cells=dict(values=list(zip(*data)))
)])

fig.update_layout(
    title=f'Score Table (10h established, MAX_SCORE={model.max_score})',
    height=350
)
fig.show()

In [ ]:
model10 = ScoreModel(max_score=10.0)
model100 = ScoreModel(max_score=100.0)

HONEST_RELAYS = 1000
HONEST_AGE = 10 * 3600
ATTACKER_CONTRIBUTIONS = 100
ATTACKER_AGE = 1 * 3600
TARGET_BW = 80

summary = []

for label, model in [('MAX_SCORE=10', model10), ('MAX_SCORE=100', model100)]:
    # Contributions to maintain cap
    cap_0h = model.contributions_to_maintain_cap(HONEST_AGE, 0)
    cap_1h = model.contributions_to_maintain_cap(HONEST_AGE, 1 * 3600)
    cap_2h = model.contributions_to_maintain_cap(HONEST_AGE, 2 * 3600)
    
    # Attack cost when honest active (100 contrib each)
    active_cost = attack_cost(model, HONEST_RELAYS, 100, HONEST_AGE, 0, ATTACKER_CONTRIBUTIONS, ATTACKER_AGE, TARGET_BW)
    
    # Attack cost when honest idle 1h (100 contrib each)
    idle1h_cost = attack_cost(model, HONEST_RELAYS, 100, HONEST_AGE, 1 * 3600, ATTACKER_CONTRIBUTIONS, ATTACKER_AGE, TARGET_BW)
    
    # Attack cost when honest idle 2h (100 contrib each)  
    idle2h_cost = attack_cost(model, HONEST_RELAYS, 100, HONEST_AGE, 2 * 3600, ATTACKER_CONTRIBUTIONS, ATTACKER_AGE, TARGET_BW)
    
    # Score difference: 100 contrib vs 10 contrib (active)
    score_100 = model.score(100, HONEST_AGE, 0)
    score_10 = model.score(10, HONEST_AGE, 0)
    differentiation = score_100 / score_10 if score_10 > 0 else float('inf')
    
    # Score after 1h idle: 100 contrib vs 10 contrib
    score_100_idle = model.score(100, HONEST_AGE, 1 * 3600)
    score_10_idle = model.score(10, HONEST_AGE, 1 * 3600)
    diff_idle = score_100_idle / score_10_idle if score_10_idle > 0 else float('inf')
    
    summary.append({
        'label': label,
        'cap_0h': cap_0h,
        'cap_1h': cap_1h,
        'cap_2h': cap_2h,
        'active_cost': active_cost['total_cost'] if active_cost else None,
        'idle1h_cost': idle1h_cost['total_cost'] if idle1h_cost else None,
        'idle2h_cost': idle2h_cost['total_cost'] if idle2h_cost else None,
        'differentiation': differentiation,
        'diff_idle': diff_idle,
    })

fig = go.Figure(data=[go.Table(
    header=dict(values=[
        'Parameter', 'MAX_SCORE=10', 'MAX_SCORE=100'
    ]),
    cells=dict(values=[
        [
            'Contrib to maintain cap (active)',
            'Contrib to maintain cap (1h idle)',
            'Contrib to maintain cap (2h idle)',
            'Attack cost (honest active)',
            'Attack cost (honest 1h idle)',
            'Attack cost (honest 2h idle)',
            'Bandwidth ratio (100 vs 10 contrib, active)',
            'Bandwidth ratio (100 vs 10 contrib, 1h idle)',
        ],
        [
            f'{summary[0]["cap_0h"]:.1f}',
            f'{summary[0]["cap_1h"]:.1f}',
            f'{summary[0]["cap_2h"]:.1f}',
            f'${summary[0]["active_cost"]:,.0f}' if summary[0]["active_cost"] else 'N/A',
            f'${summary[0]["idle1h_cost"]:,.0f}' if summary[0]["idle1h_cost"] else 'N/A',
            f'${summary[0]["idle2h_cost"]:,.0f}' if summary[0]["idle2h_cost"] else 'N/A',
            f'{summary[0]["differentiation"]:.1f}x',
            f'{summary[0]["diff_idle"]:.1f}x',
        ],
        [
            f'{summary[1]["cap_0h"]:.1f}',
            f'{summary[1]["cap_1h"]:.1f}',
            f'{summary[1]["cap_2h"]:.1f}',
            f'${summary[1]["active_cost"]:,.0f}' if summary[1]["active_cost"] else 'N/A',
            f'${summary[1]["idle1h_cost"]:,.0f}' if summary[1]["idle1h_cost"] else 'N/A',
            f'${summary[1]["idle2h_cost"]:,.0f}' if summary[1]["idle2h_cost"] else 'N/A',
            f'{summary[1]["differentiation"]:.1f}x',
            f'{summary[1]["diff_idle"]:.1f}x',
        ],
    ])
)])

fig.update_layout(title='Summary: MAX_SCORE=10 vs MAX_SCORE=100', height=400)
fig.show()

In [ ]:
from IPython.display import display, Markdown

recommendation = """
## Summary

| Metric | MAX_SCORE=10 | MAX_SCORE=100 |
|--------|--------------|---------------|
| Contrib to stay at cap (active) | 1 | 10 |
| Contrib to stay at cap (1h idle) | 4 | 40 |
| Contrib to stay at cap (2h idle) | 16 | 160 |
| Attack cost (honest active) | Same | Same |
| Attack cost (honest 1h idle) | Lower | Higher |
| Bandwidth differentiation | 1x (all at cap) | Up to 10x |

## Key Insights

1. **Attack cost when honest relays are active**: Identical for both caps - the cap cancels out in the ratio.

2. **Attack cost when honest relays are idle**: Higher MAX_SCORE provides more "buffer" against decay.
   - With MAX_SCORE=100: relays with 100 contributions stay at cap through 1h idle
   - With MAX_SCORE=10: relays with 100 contributions drop below cap after ~30min idle
   
3. **Bandwidth differentiation**: 
   - MAX_SCORE=10: Nearly all active relays hit cap → equal bandwidth share
   - MAX_SCORE=100: High-volume relays (100+ contrib) get more bandwidth than low-volume (10 contrib)

## Recommendation

**Use MAX_SCORE=100** if you want:
- Higher attack cost during quiet periods (honest relays maintain score longer)
- Bandwidth differentiation favoring high-volume relays
- Protection against "sleeper" attacks where attacker waits for honest decay

**Use MAX_SCORE=10** if you want:
- Egalitarian bandwidth allocation (all active participants equal)
- Simpler model (everyone quickly converges to cap)
- Lower barrier for new relays to reach parity

**Suggested default: MAX_SCORE=100** - provides meaningful differentiation and better decay resilience, 
while attack cost when network is active remains the same.
"""

display(Markdown(recommendation))